# [SPOT] 31. 사전등록 전향 검증: 고정 시스템을 2026년 12월 이후 자료에 한 번 적용 (해시 모드 / 분석 모드)

이 노트북은 두 번 실행합니다.
1. **해시 모드 (지금, 등록 전).** `RUN_ANALYSIS = False`. 고정된 입력(16번 모델·임계값, 26번 재학습 모델, 5·6번 기준값)의 SHA-256을 뽑아 `31_FROZEN_INPUTS.csv`에 적고,
   분석 계획 전체를 `31_PROSPECTIVE_SPEC.json`으로 저장합니다. 자료는 내려받지 않습니다. 이 노트북 파일 자체의 SHA-256과 함께 사전등록문에 넣습니다.
2. **분석 모드 (평가 기간이 끝난 뒤, 한 번).** `RUN_ANALYSIS = True`. 바이낸스 월별 1분 파일(SHA-256 검증)을 받아 아래 계획을 그대로 실행하고 잠금 파일을 만듭니다.
   잠금 파일이 있으면 다시 평가하지 않습니다. 결과는 좋든 나쁘든 그대로 보고합니다.

**평가 기간.** 2026-12-01 00:00 UTC부터 `EVAL_MONTHS`개월(기본 12). 사건은 붕괴 시각이 [시작 + 240분, 끝) 안인 것. 24개월 시점의 갱신은 서술용입니다.

**고정된 것 (바꾸지 않음).** 2021 학습·2022 보정 모델과 임계값(16번 `METHODS_FROZEN.json`), 시작점·고점 라벨 재학습 모델(26번 `ONSET_AWARE_FROZEN.json`),
달력 스케줄러(월 20·연 240·240분 쿨다운), 사건 규칙(직전 240분 고점 대비 4%, 5분 유지, 24시간 불응기; 시작점 = 고점 뒤 첫 1% 하락 종가), 판정 창(붕괴 30–240분 전).
M0는 저장된 계수·절편·표준화 상수로 다시 만들고, 캐시가 있으면 2021 재적합과 같은지 확인합니다.

**가설과 판정 (사전등록문과 같음).**
- **H1 (주, 확증).** M0의 하락 전 경고 수 > 예산 보존 국면 일치 귀무모형(26번) 기대값. 단측, α = 0.05, 10,000회. 이것 하나만 확증 검정입니다.
- **S1.** M0의 진행 중 탐지 수, 같은 귀무모형. **S2.** M1·M2·GBM 두 끝점, 끝점 안에서 Holm. **S3.** 26번 B·B2·C 모델의 하락 전 경고(같은 귀무모형), 그리고 A 대비 짝 McNemar.
- **P1 (보조 방향 예측).** 24시간 창에서 GBM − M0: 표준 AUC > 0, 하락 전 AUC < 0 (같은 분, 공유 7일 블록 부트스트랩 500회). **P2.** 4시간 창에서 하락 전 포착 사건 M0 > GBM (단측 정확 McNemar).
- **서술.** 놓친 사건의 단계 분류(신호·창·쿨다운·예산), 예산 없는 재생, 리드타임 분포. 어떤 사건도 주 검정에서 빼지 않습니다.

**스케줄러 상태.** 스케줄러는 2026-01-01부터 끊김 없이 돌립니다. 월 20·연 240 규칙 아래서는 연간 상한이 작동할 수 없지만, 서술용 '월 상한 없음' 재생은 27번에서 보았듯 11월 전에 240을 다 쓸 수 있어서
연간 누적 상태를 실제대로 복원해야 합니다. 평가 기간 전의 경보는 세지 않습니다. 연간 카운터는 2027-01-01에 초기화됩니다.

**v4 (등록 전 마지막 수정).** 사건 창 끝을 '끝 − 4분'으로 명시(5분 확인 규칙), 귀무모형에서 자리를 못 찾은 경보는 그 달 전체를 훑어 두 제약을 지키는 분으로 옮기고 위반 수를 보고, 사건 0건이어도 끝까지 실행, P2에 포착률 차이의 월 블록 부트스트랩 구간 추가.

**실패 규칙 (v3).** 사건이 없으면 H1·S1–S3·P2는 '검정 불가'로 보고합니다. 양성이나 음성이 없는 AUC는 NaN이고, 부트스트랩에서 정의되지 않은 추출은 버리되 정의된 추출이 80% 미만이면 구간을 '정의 안 됨'으로 두고 P1은 inconclusive입니다.
시드·추출 횟수·평가 기간은 결과를 본 뒤 바꾸지 않습니다. 리드타임은 포착 건마다 저장하고 30분·60분 이상 앞선 비율을 함께 냅니다. 귀무모형 재배치 품질과 월별 자료 커버리지도 보고합니다.

**상태.** 분석 코드는 자료가 생기기 전에 고정됩니다. 해시 모드 약 1분, 분석 모드 약 30–60분(자료 내려받기 포함).

In [2]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 1-7 SPOT)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, READ-ONLY
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'현물 캐시가 없습니다: {_src}. 1번 SPOT 노트북을 먼저 실행하세요.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'5·6번 SPOT 노트북 결과가 필요합니다: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 4-7)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- verbatim from original notebook (04_causal_alert_selection_code.py: emit_calendar) ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- verbatim from original notebook (04_causal_alert_selection_code.py: match_events) ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- verbatim from original notebook (04_causal_alert_selection_code.py: statistics) ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the original notebook-5 implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from original at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook-5 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook-1 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- verbatim from original notebook (01_cache_77months_code.py: chunked_hs) ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT 계산 {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook-1 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- verbatim from original notebook (06_validation_sourcefix_v2_code.py: price_only_catalog) ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
# Shared by notebooks 18 and 19: exact block-bootstrap AUC decomposition.
def week_pairs(sp, wp, sn, wn):
    """C[k, l] = over positives in block k, the number of negatives in block l with a lower score (+0.5 per tie).
    With block multiplicities w from the bootstrap, the AUC is w'Cw / ((w'P)(w'N)); w = 1 gives the ordinary AUC."""
    C = np.zeros((N_WEEKS, N_WEEKS))
    if len(sp) == 0 or len(sn) == 0:
        return C
    o = np.lexsort((sn, wn)); sn = sn[o]; wn = wn[o]
    bounds = np.searchsorted(wn, np.arange(N_WEEKS + 1), 'left')
    for l in range(N_WEEKS):
        a, b = bounds[l], bounds[l + 1]
        if a == b:
            continue
        seg = sn[a:b]; lo = np.searchsorted(seg, sp, 'left'); hi = np.searchsorted(seg, sp, 'right')
        C[:, l] = np.bincount(wp, weights=lo + 0.5 * (hi - lo), minlength=N_WEEKS)
    return C


def _ratio(a, b):
    return a / b if b > 0 else np.nan


STAT_KEYS = ['auc_standard', 'auc_pre', 'auc_in', 'share_in', 'diff_standard_minus_pre', 'in_share_of_excess', 'auc_calm',
             'auc_standard_within_regime', 'auc_pre_within_regime', 'auc_in_within_regime']


def make_parts(sm, code, calm, wk, dec):
    neg = code == 0; pre = code == 1; inn = code == 2; cnt = lambda k: np.bincount(wk[k], minlength=N_WEEKS).astype(float)
    P = dict(C_pre=week_pairs(sm[pre], wk[pre], sm[neg], wk[neg]), C_in=week_pairs(sm[inn], wk[inn], sm[neg], wk[neg]),
             C_calm=week_pairs(sm[(code > 0) & calm], wk[(code > 0) & calm], sm[neg & calm], wk[neg & calm]),
             P_pre=cnt(pre), P_in=cnt(inn), N=cnt(neg), P_calm=cnt((code > 0) & calm), N_calm=cnt(neg & calm),
             Cs_pre=np.zeros((N_WEEKS, N_WEEKS)), Cs_in=np.zeros((N_WEEKS, N_WEEKS)), Pq_pre=[], Pq_in=[], Nq=[])
    for q in range(10):
        kq = dec == q; nq = neg & kq; pq = pre & kq; iq = inn & kq
        P['Cs_pre'] += week_pairs(sm[pq], wk[pq], sm[nq], wk[nq]); P['Cs_in'] += week_pairs(sm[iq], wk[iq], sm[nq], wk[nq])
        P['Pq_pre'].append(cnt(pq)); P['Pq_in'].append(cnt(iq)); P['Nq'].append(cnt(nq))
    for k in ('Pq_pre', 'Pq_in', 'Nq'):
        P[k] = np.array(P[k])
    return P


def stats_from(P, w):
    q = lambda C: w @ C @ w
    Wn = w @ P['N']; Wp = w @ P['P_pre']; Wi = w @ P['P_in']; sp = q(P['C_pre']); si = q(P['C_in'])
    std = _ratio(sp + si, (Wp + Wi) * Wn); pre = _ratio(sp, Wp * Wn); inn = _ratio(si, Wi * Wn); pi = _ratio(Wi, Wp + Wi)
    nq = P['Nq'] @ w; pq = P['Pq_pre'] @ w; iq = P['Pq_in'] @ w; ssp = q(P['Cs_pre']); ssi = q(P['Cs_in'])
    return dict(auc_standard=std, auc_pre=pre, auc_in=inn, share_in=pi, diff_standard_minus_pre=std - pre,
                in_share_of_excess=_ratio(pi * (inn - 0.5), std - 0.5) if np.isfinite(inn) else 0.0,
                auc_calm=_ratio(q(P['C_calm']), (w @ P['P_calm']) * (w @ P['N_calm'])),
                auc_standard_within_regime=_ratio(ssp + ssi, ((pq + iq) * nq).sum()),
                auc_pre_within_regime=_ratio(ssp, (pq * nq).sum()), auc_in_within_regime=_ratio(ssi, (iq * nq).sum()))
import joblib
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import HistGradientBoostingClassifier

RUN_ANALYSIS = False                 # False: hash mode (no data). True: the one-shot analysis after the evaluation period ends.
EVAL_MONTHS = 12                     # pre-registered evaluation horizon in months from PROS_START
PROS_START = ts('2026-12-01')
PROS_END = PROS_START + pd.DateOffset(months=EVAL_MONTHS)
EV_END = PROS_END - pd.Timedelta(minutes=4)   # a breach needs 5 closes (itself + 4) to be confirmed, so a breach in the last 4 minutes cannot be confirmed from the period's own archives
WARMUP_MONTHS_FIRST = '2025-12'      # archives from here: 30-day feature warm-up before the scheduler starts on 2026-01-01, and the overlap with the spot cache (through 2026-04-30) for the guard
SCHED_START = ts('2026-01-01')       # the scheduler runs from the start of the calendar year, so the annual count, monthly count and cooldown at PROS_START are the system's real state; alerts before PROS_START are not counted
OUT = SPOT_BASE / 'prospective_test_v1'; OUT.mkdir(parents=True, exist_ok=True)
RAW31 = DRIVE / 'BTCUSDT_spot_prospective_2026_2027' / '1m'
M16 = SPOT_BASE / 'methods_definitions_v1'; M26 = SPOT_BASE / 'onset_aware_training_v1'
DNAME, DCL = '4pct', 0.04
METHODS16 = ['M0_VF_logit', 'M1_vol_threshold', 'M2_VPIN', 'M3_GBM_rich']
VARIANTS26 = ['B_pre_onset', 'B2_pre_onset_in_decline_negative', 'C_pre_peak']
H_P1, H_P2, H_MAIN = 1440, 240, 240
N_NULL = 10000; N_BOOT = 500; BLOCK_DAYS = 7; BP_TRIES = 64; ALPHA = 0.05; SEED = 20261101
MIN_DEFINED_SHARE = 0.8              # a bootstrap interval is reported only if the quantity is defined in at least this share of draws; otherwise undefined -> inconclusive
LEAD_MARKS = [30, 60]                # lead-time marks (minutes before the onset) for the descriptive lead-time table
LOCK = OUT / 'PROSPECTIVE_EVALUATED.lock'

FROZEN_FILES = {'16_METHODS_FROZEN': M16 / 'METHODS_FROZEN.json', '16_METHODS_SPEC': M16 / 'METHODS_SPEC.json',
                '26_ONSET_AWARE_FROZEN': M26 / 'ONSET_AWARE_FROZEN.json', '26_SPEC': M26 / '26_SPEC.json',
                '05_spot_reference': REF_PATH, '06_spot_frozen_thresholds': THR_PATH,
                '02_price_only_event_catalog': SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv'}
for k, p in FROZEN_FILES.items():
    if not p.is_file():
        raise FileNotFoundError(f'고정 입력이 없습니다 ({k}): {p}')
FROZEN16 = json.loads(FROZEN_FILES['16_METHODS_FROZEN'].read_text()); SPEC16 = json.loads(FROZEN_FILES['16_METHODS_SPEC'].read_text())
FROZEN26 = json.loads(FROZEN_FILES['26_ONSET_AWARE_FROZEN'].read_text())
if sha256_file(FROZEN_FILES['16_METHODS_SPEC']) != FROZEN16['spec_sha256'] or SPEC16['gbm_params'] != GBM_PARAMS or SPEC16['gbm_features'] != FEATS:
    raise RuntimeError('GUARD FAIL: notebook-16 spec/settings differ')
CFG = {m: FROZEN16['configs'][f'{m}|{DNAME}'] for m in METHODS16}
CFG.update({f'{m}|{lab}': FROZEN26['configs'][f'{m}|{lab}'] for m in ['M0_VF_logit', 'M3_GBM_rich'] for lab in VARIANTS26})
for key, c in CFG.items():
    if c.get('model_file'):
        FROZEN_FILES[f'model_{key}'] = SPOT_BASE / c['model_file']
        if not FROZEN_FILES[f'model_{key}'].is_file():
            raise FileNotFoundError(f'고정 모델 파일이 없습니다: {c["model_file"]}')
        if sha256_file(FROZEN_FILES[f'model_{key}']) != c['model_sha256']:
            raise RuntimeError(f'GUARD FAIL: model file hash differs from its freeze record ({key})')

SPEC = {'title': 'Pre-registered prospective test of frozen crash-warning systems, Binance spot BTCUSDT 1-minute, from 2026-12-01',
        'evaluation_period': {'start_utc': str(PROS_START), 'end_utc': str(PROS_END), 'months': EVAL_MONTHS,
                              'events': 'breach in [start + 240 min, end - 4 min): the 5-close confirmation needs 4 closes after the breach, and the archives of the period end at end - 1 min; '
                                        'monthly archives only, SHA-256 verified; no API data in the confirmatory run',
                              'descriptive_update': 'the same analysis at 24 months, reported as descriptive'},
        'frozen': {'systems': 'notebook-16 METHODS_FROZEN.json (M0 logistic from stored coefficients and scaler; M1 z_vol; M2 VPIN; M3 GBM file); '
                              'notebook-26 ONSET_AWARE_FROZEN.json (M0 and GBM retrained on labels B, B2, C); thresholds calibrated on 2022; nothing refitted',
                   'scheduler': f'calendar: {MONTH_CAP}/month, {B}/calendar year, {COOLDOWN_MIN}-min cooldown, run continuously from {SCHED_START.date()} so that the annual count, the monthly count and the cooldown '
                                'at the start of the period are the real state of a system that has been running all year (under the monthly cap the annual cap cannot bind, but the '
                                'no-monthly-cap replay can exhaust 240 before November, as notebook 27 found for 2026); alerts before the period are not counted; the annual count resets on 2027-01-01',
                   'events': 'price_only_catalog: 4% below the prior 240-min high, 5 closes sustained, 1440-min refractory; onset = first close >= 1% below the peak',
                   'judging_window': [LEAD_MIN, LEAD_MAX], 'warm_up': f'archives from {WARMUP_MONTHS_FIRST} for the 30-day feature baselines and the cache guard',
                   'secondary_event_definitions': {'P1_H1440': 'price_only_catalog with H = 1440: peak = highest close in the 1440 minutes before the breach, breach = close <= 0.96 x that peak, '
                                                                'sustained 5 closes, 1440-min refractory; onset = first close after the peak at least 1% below it; per-minute label = breach 30-240 min ahead; '
                                                                'in-progress positive = at or after the onset of the event that makes the minute positive; evaluation minutes = valid minutes in the period '
                                                                'ending 245 min before its end with finite 24h RV, both M0 and GBM scores finite; RV deciles and 7-day blocks over those minutes',
                                                   'P2_H240': 'the main definition (H = 240) and the frozen alerts; events caught pre-onset per method by the notebook-4 matcher'},
                   'code': 'all analysis code (archive download and checksum, feature recomputation, catalog rule, calendar scheduler and its Python replay, notebook-4 matcher, '
                           'budget-preserving null, block-bootstrap AUC decomposition, stage classification) is embedded in this notebook, so the SHA-256 of the .ipynb file recorded in the '
                           'registration covers it; functions originate in notebooks 1, 2, 4, 5, 16, 18, 22, 26 and 27 of the replication package'},
        'guards': ['downloaded prices and recomputed signals equal the spot cache on the overlap (notebook-11 tolerances)',
                   'M0 rebuilt from stored parameters equals a 2021 refit when the cache is present', 'model file hashes equal their freeze records',
                   'compiled scheduler equals the Python replay'],
        'H1_primary': {'claim': 'M0 (label A) pre-onset catches exceed the budget-preserving regime-matched null', 'null': 'notebook-26 draw_budget',
                       'test': f'one-sided, p = (1 + #draws >= observed) / ({N_NULL} + 1), alpha {ALPHA}', 'decision': 'supported if p < alpha; reported either way'},
        'S1': 'M0 in-progress catches under the same null (nominal p)',
        'S2': 'M1, M2, M3 on both endpoints under the same null; Holm across the three methods within each endpoint',
        'S3': 'labels B, B2, C for M0 and GBM: pre-onset catches under the same null; exact one-sided McNemar (variant better than A) on pre-onset catches, Holm within method',
        'P1': {'claim': 'at H = 1440: GBM - M0 standard AUC > 0 and GBM - M0 pre-onset AUC < 0', 'inference': f'shared {N_BOOT} x {BLOCK_DAYS}-day block bootstrap on the common minutes; '
               'read supported if both intervals exclude 0 in the predicted directions, contradicted if either excludes 0 in the other direction, inconclusive otherwise; within-regime pre-onset AUC also reported; '
               f'a difference is undefined in a draw when either AUC has no positive or no negative minute, and an interval is reported only when the difference is defined in at least {MIN_DEFINED_SHARE:.0%} of draws, '
               'otherwise it is undefined and P1 is inconclusive; the number of defined draws is reported'},
        'P2': {'claim': 'at H = 240: M0 catches more events before onset than GBM under the frozen alerts', 'inference': 'exact one-sided McNemar on discordant events; event counts reported; '
               'inconclusive if fewer than 10 events are caught pre-onset by either method; the paired difference in pre-onset catch rate (M0 minus GBM) is reported with a 95% interval '
               f'from a calendar-month block bootstrap over events ({N_NULL} draws, same defined-draw rule as P1); the interval describes the size of the effect and does not decide P2'},
        'descriptive': ['stage decomposition of missed events (signal / window / cooldown / budget) with the Python replay', 'no-monthly-cap and cooldown-only replays',
                        f'lead times of pre-onset catches per event (saved), with the median and the shares of catches at least {LEAD_MARKS} minutes before the onset and within 60 minutes of it',
                        'null-draw audit: share of alerts placed in their own volatility decile, in the nearest decile, or left in place; monthly-count, cap, budget and cooldown checks on the first 50 draws',
                        'data coverage by month: minutes on the grid, minutes missing from the archives, invalid minutes; events whose 240-min warning window crosses the period start are excluded by the event rule and counted',
                        'lead-time distribution of pre-onset catches', 'all alerts and events listed with timestamps (no prices)'],
        'exclusions': 'none; minutes with missing archive data are NaN and cannot host alerts or events; exchange incidents are reported, not excluded',
        'failure_rules': {'no_events': 'if no evaluation event exists, H1, S1, S2, S3 and P2 are reported as not testable; nothing is extended or re-drawn',
                          'null_distribution': 'the null p-value is defined for any number of events (including zero catches); it is never re-drawn',
                          'null_placement': 'decile search (same, then nearest deciles, 64 candidates each), then a scan of every minute of the month for one that keeps both the 240-min distance from the original '
                                            'and the cooldown against alerts already placed; an alert with no feasible minute keeps its time and the draw is kept; the numbers of such alerts and of the '
                                            'cooldown violations they create are reported over all draws (expected to be zero)',
                          'undefined_auc': 'an AUC with no positive or no negative minute is NaN; a NaN point estimate makes P1 inconclusive',
                          'bootstrap': f'draws in which a quantity is undefined are dropped from its interval; if fewer than {MIN_DEFINED_SHARE:.0%} of draws define it, the interval is undefined and the reading is inconclusive',
                          'no_resampling_or_extension': 'the seed, the number of draws and the evaluation period are fixed here and are not changed after any result is seen'},
        'draws': N_NULL, 'bootstrap': N_BOOT, 'block_days': BLOCK_DAYS, 'bp_tries': BP_TRIES, 'alpha': ALPHA, 'seed': SEED,
        'frozen_input_sha256': {k: sha256_file(p) for k, p in sorted(FROZEN_FILES.items())},
        'rules': 'all outcomes reported as observed; no rule, threshold or model changed after the data exist; second evaluation refused (lock file)'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '31_PROSPECTIVE_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    if RUN_ANALYSIS:
        raise RuntimeError('기존 31_PROSPECTIVE_SPEC.json과 다릅니다. 등록된 계획을 바꿔 분석할 수 없습니다.')
    _sp.rename(OUT / f'31_PROSPECTIVE_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
MANI = pd.DataFrame([dict(input=k, path=str(p.relative_to(DRIVE)), sha256=sha256_file(p), bytes=p.stat().st_size) for k, p in sorted(FROZEN_FILES.items())]
                    + [dict(input='31_PROSPECTIVE_SPEC', path=str(_sp.relative_to(DRIVE)), sha256=SPEC_SHA, bytes=len(_raw))])
MANI.to_csv(OUT / '31_FROZEN_INPUTS.csv', index=False)
print('Output:', OUT, '| spec sha256', SPEC_SHA); display(MANI)
TH = {k: float(c['threshold_2022']) for k, c in CFG.items()}
print('Frozen thresholds:', {k: round(v, 6) for k, v in TH.items()})
if not RUN_ANALYSIS:
    print('\nHASH MODE: the table above goes into the registration. Set RUN_ANALYSIS = True only after', PROS_END.date(), 'and run once.')

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/prospective_test_v1 | spec sha256 ebedf6ea4e4a53891237b57346e015fafe062295c52c5c0c83a4af2c11b2c3e2


,input,path,sha256,bytes
0,02_price_only_event_catalog,btc_lead_cost_results_77months_spot/event_defi...,9c9c035fda635b75f5f19598f34bb52dd55bbfe4b93c11...,42280
1,05_spot_reference,btc_lead_cost_results_77months_spot/causal_ale...,df54cddf8263cfa8bdd1b5483d891c32ff7d25f3f622ce...,489
2,06_spot_frozen_thresholds,btc_lead_cost_results_77months_spot/FINAL_risk...,faef731db6f7612faf2adc8a01e6c5cffbcfc3515b6c43...,263
3,16_METHODS_FROZEN,btc_lead_cost_results_77months_spot/methods_de...,2db183557433f15544c024b44761dd6665c36d39852c0b...,3793
4,16_METHODS_SPEC,btc_lead_cost_results_77months_spot/methods_de...,23e3f879a7738de2dce272cf0d6b0646d2c74fddcc94d4...,2455
5,26_ONSET_AWARE_FROZEN,btc_lead_cost_results_77months_spot/onset_awar...,63d08c6e8e37515feef7843f2f8d4060b1304f2a3336d8...,3719
6,26_SPEC,btc_lead_cost_results_77months_spot/onset_awar...,b061b78cb73807b0441c0a87722bc7b79bf39d7ac78ab7...,2888
7,model_M3_GBM_rich,btc_lead_cost_results_77months_spot/methods_de...,176fdac46ff63729ed8f271540b356543a6e6bf1e977ed...,1118448
8,model_M3_GBM_rich|B2_pre_onset_in_decline_nega...,btc_lead_cost_results_77months_spot/onset_awar...,ca4ae55deeac65c0deebcb13d3536f68118c74ce0d78b7...,1118480
9,model_M3_GBM_rich|B_pre_onset,btc_lead_cost_results_77months_spot/onset_awar...,a36d4d2894dbc4558da07d0bbbae5dad40bc1834786128...,1118480


Frozen thresholds: {'M0_VF_logit': 0.669899, 'M1_vol_threshold': 1.676028, 'M2_VPIN': 0.036001, 'M3_GBM_rich': 0.113531, 'M0_VF_logit|B_pre_onset': 0.59639, 'M0_VF_logit|B2_pre_onset_in_decline_negative': 0.589926, 'M0_VF_logit|C_pre_peak': 0.625673, 'M3_GBM_rich|B_pre_onset': 0.091018, 'M3_GBM_rich|B2_pre_onset_in_decline_negative': 0.095587, 'M3_GBM_rich|C_pre_peak': 0.114367}

HASH MODE: the table above goes into the registration. Set RUN_ANALYSIS = True only after 2027-12-01 and run once.


## 1. 자료 (분석 모드만): 월별 파일 내려받기, 캐시 대조, 고정 모델 재현

마지막 평가 월의 월별 파일이 아직 없으면 멈춥니다(보통 다음 달 첫 주에 올라옵니다). 일별 파일이나 API 자료로 대신하지 않습니다.

In [3]:
def logit_from_params(c):
    """Rebuild a standardized logistic predictor from the stored notebook-16/26 parameters (no refit)."""
    mu = np.asarray(c['scaler_mean'], float); sc = np.asarray(c['scaler_scale'], float); w = np.asarray(c['coef'], float); b0 = float(c['intercept'])
    def f(X):
        z = ((X - mu) / sc) @ w + b0
        return 1.0 / (1.0 + np.exp(-z))
    return f


if RUN_ANALYSIS:
    if LOCK.exists():
        raise RuntimeError(f'이 전향 검증은 이미 평가됐습니다 (one-shot). 결과: {OUT}. 다시 평가하지 않습니다.')
    if pd.Timestamp.now(tz='UTC') < PROS_END:
        raise RuntimeError(f'평가 기간이 아직 끝나지 않았습니다 ({PROS_END}). 그 뒤에 한 번 실행하세요.')
    t_ = time.time()
    last_month = (PROS_END - pd.Timedelta(minutes=1)).strftime('%Y-%m')
    MONTHS = month_list(WARMUP_MONTHS_FIRST, last_month)
    man = get_archives('spot', 'BTCUSDT', MONTHS, RAW31)          # raises if any month is not published yet
    man.to_csv(OUT / '0_download_manifest.csv', index=False)
    raw = load_archives(man, RAW31)
    px = regularize(raw, start=ts(WARMUP_MONTHS_FIRST + '-01'), end=PROS_END + pd.Timedelta(days=1) - pd.Timedelta(minutes=1))
    px = px.loc[:min(px.index.max(), raw.index.max())]
    sg = compute_spot_features(px)
    # Guard against the spot cache on the overlap (same checks and tolerances as notebooks 11 and 27)
    pxc, sgc = load_btc_spot()
    ov = pxc.index[(pxc.index >= px.index.min() + pd.Timedelta(days=31)) & (pxc.index <= min(pxc.index.max(), px.index.max()))]
    if len(ov) < 30 * 1440:
        raise RuntimeError('GUARD FAIL: too little overlap with the spot cache to verify the signal pipeline')
    for col in ['close', 'volume', 'taker_buy_vol']:
        a_ = px.loc[ov, col].to_numpy(float); b_ = pxc.loc[ov, col].to_numpy(float); both = np.isfinite(a_) & np.isfinite(b_)
        if (np.isfinite(a_) != np.isfinite(b_)).any() or not np.array_equal(a_[both], b_[both]):
            raise RuntimeError(f'GUARD FAIL: archive {col} differs from the spot cache on the overlap')
    for col, tol in [('z_vol', 5e-4), ('tsr', 5e-6), ('z_hs', 5e-4)]:
        a_ = sg.loc[ov, col].to_numpy(float); b_ = sgc.loc[ov, col].to_numpy(float); both = np.isfinite(a_) & np.isfinite(b_)
        md = float(np.max(np.abs(a_[both] - b_[both]))) if both.any() else np.inf
        if (np.isfinite(a_) != np.isfinite(b_)).mean() > 1e-4 or md > tol:
            raise RuntimeError(f'GUARD FAIL: recomputed {col} differs from the spot cache (max |diff| {md:.3g})')
    # M0 from stored parameters must equal a 2021 refit on the cache
    Tc = pxc.index.asi8; vc = sgc.z_vol.to_numpy(float); fc = sgc.tsr.to_numpy(float); hc = sgc.z_hs.to_numpy(float)
    Vc = (np.isfinite(pxc.close.to_numpy()) & (pxc.close.to_numpy() > 0) & np.isfinite(pxc.volume.to_numpy()) & (pxc.volume.to_numpy() > 0)
          & np.isfinite(vc) & np.isfinite(fc) & np.isfinite(hc) & (fc >= 0) & (fc <= 1))
    Ec = load_catalog(FROZEN_FILES['02_price_only_event_catalog']); Xc = np.column_stack([vc, fc]); okl = train_mask(Tc, Vc) & np.isfinite(Xc).all(axis=1)
    m0 = new_logit().fit(Xc[okl], label_from_catalog(Tc, Ec)[okl]); f0 = logit_from_params(CFG['M0_VF_logit'])
    smp = np.flatnonzero(Vc & np.isfinite(Xc).all(axis=1))[::997]
    d0 = float(np.abs(m0.predict_proba(Xc[smp])[:, 1] - f0(Xc[smp])).max())
    if d0 > 1e-9:
        raise RuntimeError(f'GUARD FAIL: M0 from stored parameters differs from the 2021 refit (max |diff| {d0:.2e})')
    del pxc, sgc, Tc, vc, fc, hc, Vc, Xc, m0
    print(f'PASS: prices, signals and M0 verified against the spot cache on {ov[0]:%Y-%m-%d} -> {ov[-1]:%Y-%m-%d} ({time.time() - t_:.0f}s)')
    print(f'price grid {px.index.min()} -> {px.index.max()} ({len(px):,} minutes, {int(px.close.isna().sum())} NaN closes)')

## 2. 점수, 경보, 사건 (분석 모드만)

In [4]:
if RUN_ANALYSIS:
    t_ = time.time()
    T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
    v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
    VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
    F = causal_features(px, sg); X_all = F[FEATS].to_numpy(np.float32); Xl = np.column_stack([v_, f_])
    S = {}
    for key, c in CFG.items():
        s = np.full(len(T), np.nan)
        if key == 'M1_vol_threshold':
            s[VALID] = F.z_vol.to_numpy(float)[VALID]
        elif key == 'M2_VPIN':
            s[VALID] = F.vpin.to_numpy(float)[VALID]
        elif key.startswith('M0_VF_logit'):
            okp = VALID & np.isfinite(Xl).all(axis=1); s[okp] = logit_from_params(c)(Xl[okp])
        else:
            s[VALID] = joblib.load(SPOT_BASE / c['model_file']).predict_proba(X_all[VALID])[:, 1]
        S[key] = s
    SCH = VALID & (T >= SCHED_START.value)
    ALR = {}
    for key in S:
        ok_ = SCH & np.isfinite(S[key]); a = emit_calendar_fast(T[ok_], S[key][ok_] >= TH[key])
        ALR[key] = a[a >= PROS_START.value]
        pd.DataFrame({'alarm_utc': pd.to_datetime(ALR[key], utc=True)}).to_csv(OUT / f'alarms_{key.replace("|", "_")}.csv', index=False)
    E_ALL = {}
    for H in sorted({H_MAIN, H_P1, H_P2}):
        try:
            E_ = price_only_catalog(close, volume, px.index, H=H, decline=DCL)
        except RuntimeError as e_:            # the frozen rule raises when the whole series has no event; keep an empty catalog and continue
            if 'No price-only crash events' not in str(e_):
                raise
            E_ = pd.DataFrame(columns=['event_id', 'peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc', 'peak_to_breach_minutes', 'onset_to_breach_minutes', 'year'])
        for c_ in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
            E_[c_] = pd.to_datetime(E_[c_], utc=True)
        E_ALL[H] = E_.sort_values('breach_utc').reset_index(drop=True)
    EV = {H: events_between(E_ALL[H], PROS_START.strftime('%Y-%m-%d %H:%M'), EV_END.strftime('%Y-%m-%d %H:%M')) for H in E_ALL}
    for H, ev in EV.items():
        ev[['event_id', 'peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']].to_csv(OUT / f'1_events_H{H}.csv', index=False)
    print('alerts in the evaluation period:', {k: len(v) for k, v in ALR.items()})
    print('events:', {H: len(ev) for H, ev in EV.items()}, f'({time.time() - t_:.0f}s)')

## 3. 예산 보존 국면 일치 귀무모형 (26번과 같은 코드) — H1, S1, S2, S3

In [5]:
def ev_arrays(ev):
    return dict(br=ev.breach_utc.astype('int64').to_numpy(), on=ev.onset_1pct_utc.astype('int64').to_numpy(), pk=ev.peak_utc.astype('int64').to_numpy(), n=len(ev))


def fast_counts(a, ea):
    a = np.sort(np.asarray(a, np.int64)); used = np.zeros(len(a), bool); pre = post = 0; leads = []
    for k in range(ea['n']):
        b = ea['br'][k]; l_ = np.searchsorted(a, b - LEAD_MAX * NS, 'left'); r_ = np.searchsorted(a, b - LEAD_MIN * NS, 'right')
        for j in range(l_, r_):
            if not used[j]:
                used[j] = True; t = a[j]
                if t < ea['on'][k]:
                    pre += 1; leads.append((ea['on'][k] - t) / NS)
                else:
                    post += 1
                break
    return dict(pre=pre, post=post, leads=leads)


DQ_ORDER = [sorted(range(10), key=lambda d, q=q: (abs(d - q), d)) for q in range(10)]


def make_space(Tp, RVp, start, end):
    okd = (Tp >= start.value) & (Tp < end.value) & np.isfinite(RVp)
    tt = Tp[okd]; dq = np.minimum((pd.Series(RVp[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    mo = cal_fields(tt)[1]; key = mo * 10 + dq; o2 = np.lexsort((tt, key))
    return dict(TT=tt, DD=dq, PK=tt[o2], KS=key[o2])


def draw_budget(a, rng, sp, audit=None):
    """Budget-preserving regime-matched draw (notebook 26, with one addition): same calendar month, same (or nearest) RV decile, >= 240 min from the original
    and from every alert already placed. If the decile search fails, every minute of the month is scanned and a random feasible one is used (audit: any_decile).
    Only if no feasible minute exists does the alert keep its time; that case is counted (unplaced) and any cooldown violation it creates is counted too."""
    import bisect
    a = np.asarray(a, np.int64); j = np.clip(np.searchsorted(sp['TT'], a, 'right') - 1, 0, len(sp['TT']) - 1)
    q0 = sp['DD'][j]; mo = cal_fields(a)[1]; KS, PK = sp['KS'], sp['PK']
    placed = []; out = np.empty(len(a), np.int64); exact = near = anyq = failed = viol = 0
    for i in rng.permutation(len(a)):
        ai = int(a[i]); done = False
        for dq in DQ_ORDER[q0[i]]:
            key = mo[i] * 10 + dq; lo = np.searchsorted(KS, key, 'left'); hi = np.searchsorted(KS, key, 'right')
            if hi <= lo:
                continue
            for c in PK[lo + rng.integers(0, hi - lo, BP_TRIES)]:
                c = int(c)
                if abs(c - ai) < LEAD_MAX * NS:
                    continue
                k = bisect.bisect_left(placed, c)
                if (k > 0 and c - placed[k - 1] < COOLDOWN * NS) or (k < len(placed) and placed[k] - c < COOLDOWN * NS):
                    continue
                placed.insert(k, c); out[i] = c; done = True; break
            if done:
                exact += int(dq == q0[i]); near += int(dq != q0[i]); break
        if not done:
            lo = np.searchsorted(KS, mo[i] * 10, 'left'); hi = np.searchsorted(KS, mo[i] * 10 + 9, 'right'); cand = PK[lo:hi]
            cand = cand[np.abs(cand - ai) >= LEAD_MAX * NS]
            if len(placed) and len(cand):
                pl = np.asarray(placed); kk = np.searchsorted(pl, cand)
                left_ok = np.where(kk > 0, cand - pl[np.maximum(kk - 1, 0)] >= COOLDOWN * NS, True)
                right_ok = np.where(kk < len(pl), pl[np.minimum(kk, len(pl) - 1)] - cand >= COOLDOWN * NS, True)
                cand = cand[left_ok & right_ok]
            if len(cand):
                c = int(cand[rng.integers(0, len(cand))]); bisect.insort(placed, c); out[i] = c; anyq += 1
            else:
                failed += 1; out[i] = ai
                k = bisect.bisect_left(placed, ai)
                if (k > 0 and ai - placed[k - 1] < COOLDOWN * NS) or (k < len(placed) and placed[k] - ai < COOLDOWN * NS):
                    viol += 1
                placed.insert(k, ai)
    if audit is not None:
        audit['exact'] += exact; audit['near'] += near; audit['any'] += anyq; audit['failed'] += failed; audit['viol'] += viol; audit['n'] += len(a)
    return np.sort(out)


def constraint_check(a_null, a_ref):
    """Monthly counts vs the real alerts, months over the cap, years over the budget, gaps under the cooldown (notebook 26)."""
    mr = pd.Series(cal_fields(a_ref)[1]).value_counts(); mn = pd.Series(cal_fields(a_null)[1]).value_counts()
    yn = pd.Series(cal_fields(a_null)[0]).value_counts(); u = mr.index.union(mn.index)
    return dict(months_count_changed=int((mn.reindex(u, fill_value=0) != mr.reindex(u, fill_value=0)).sum()), months_over_cap=int((mn > MONTH_CAP).sum()),
                years_over_budget=int((yn > B).sum()), gaps_under_cooldown=int((np.diff(np.sort(a_null)) < COOLDOWN * NS).sum()))


def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); m = len(p); run = 0.0
    for r, i in enumerate(o):
        run = max(run, (m - r) * p[i]); adj[i] = min(1.0, run)
    return adj


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p = .5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


if RUN_ANALYSIS:
    t_ = time.time()
    _lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
    RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
    SP = make_space(T, np.where(VALID, RV, np.nan), PROS_START, PROS_END)
    EA = ev_arrays(EV[H_MAIN])
    # fast matcher must equal the notebook-4 matcher
    for key in ['M0_VF_logit', 'M3_GBM_rich']:
        mm = match_events(EV[H_MAIN], ALR[key]); fc_ = fast_counts(ALR[key], EA)
        if (int(mm.pre1pct.sum()), int((~mm.pre1pct).sum())) != (fc_['pre'], fc_['post']):
            raise RuntimeError('GUARD FAIL: fast matcher differs from match_events')
    NO_EVENTS = EA['n'] == 0
    if NO_EVENTS:
        print('No evaluation event under the main definition: H1, S1, S2, S3 and P2 are not testable; counts and nulls are still written.')
    rows = []; LEADS = {}
    for si, key in enumerate(S):
        a = ALR[key]; obs = fast_counts(a, EA); LEADS[key] = obs['leads']
        rng = np.random.default_rng(SEED + si); dist_pre = np.empty(N_NULL); dist_post = np.empty(N_NULL); aud = dict(exact=0, near=0, any=0, failed=0, viol=0, n=0); cc = []
        for i in range(N_NULL):
            sh = draw_budget(a, rng, SP, aud); c = fast_counts(sh, EA); dist_pre[i] = c['pre']; dist_post[i] = c['post']
            if i < 50:
                cc.append(constraint_check(sh, a))
        ccd = pd.DataFrame(cc) if cc else pd.DataFrame()
        rows.append(dict(system=key, alerts=len(a), events=EA['n'],
                         pre=obs['pre'], pre_null_mean=float(dist_pre.mean()), pre_null_q95=float(np.quantile(dist_pre, 0.95)),
                         p_pre=(1 + int((dist_pre >= obs['pre']).sum())) / (N_NULL + 1), pre_excess_per_100_alerts=100 * (obs['pre'] - dist_pre.mean()) / max(1, len(a)),
                         post=obs['post'], post_null_mean=float(dist_post.mean()), p_post=(1 + int((dist_post >= obs['post']).sum())) / (N_NULL + 1),
                         post_excess_per_100_alerts=100 * (obs['post'] - dist_post.mean()) / max(1, len(a)),
                         share_same_decile=aud['exact'] / max(1, aud['n']), share_nearest_decile=aud['near'] / max(1, aud['n']), share_any_decile=aud['any'] / max(1, aud['n']),
                         share_unplaced=aud['failed'] / max(1, aud['n']), cooldown_violations_total=int(aud['viol']),
                         **{f'{k}_mean_first50': float(ccd[k].mean()) for k in ccd.columns}))
        print(f'{key:40s} alerts {len(a):4d} | pre {obs["pre"]:3d} vs null {dist_pre.mean():6.2f} (p {rows[-1]["p_pre"]:.4f}) | in-progress {obs["post"]:3d} vs {dist_post.mean():6.2f} (p {rows[-1]["p_post"]:.4f}) ({time.time() - t_:.0f}s)', flush=True)
    NT = pd.DataFrame(rows)
    NT['role'] = np.where(NT.system == 'M0_VF_logit', 'H1 primary (pre) / S1 (post)', np.where(NT.system.isin(METHODS16), 'S2', 'S3'))
    s2 = NT.system.isin(['M1_vol_threshold', 'M2_VPIN', 'M3_GBM_rich'])
    NT.loc[s2, 'p_pre_holm'] = holm(NT.loc[s2, 'p_pre']); NT.loc[s2, 'p_post_holm'] = holm(NT.loc[s2, 'p_post'])
    H1_SUPPORTED = (None if NO_EVENTS else bool(NT.loc[NT.system == 'M0_VF_logit', 'p_pre'].iloc[0] < ALPHA))
    # S3: variant vs its label-A counterpart, exact McNemar on pre-onset catches (which events)
    def pre_set(key):
        mm = match_events(EV[H_MAIN], ALR[key]); return set(mm.loc[mm.pre1pct, 'event_id'])
    s3 = []
    for m in ['M0_VF_logit', 'M3_GBM_rich']:
        pa = pre_set(m)
        for lab in VARIANTS26:
            pv = pre_set(f'{m}|{lab}')
            s3.append(dict(method=m, label=lab, A_pre=len(pa), variant_pre=len(pv), A_only=len(pa - pv), variant_only=len(pv - pa),
                           p_variant_better=mcnemar_one_sided(len(pa - pv), len(pv - pa))))
    S3T = pd.DataFrame(s3)
    for m in ['M0_VF_logit', 'M3_GBM_rich']:
        k_ = S3T.method == m; S3T.loc[k_, 'p_holm'] = holm(S3T.loc[k_, 'p_variant_better'])
    NT.to_csv(OUT / '2_null_tests.csv', index=False); S3T.to_csv(OUT / '3_onset_aware_vs_A.csv', index=False)
    display(NT.round(4)); display(S3T.round(4))
    print(f'H1 (primary): M0 pre-onset p = {NT.loc[NT.system == "M0_VF_logit", "p_pre"].iloc[0]:.4f} -> {"NOT TESTABLE (no events)" if NO_EVENTS else ("SUPPORTED" if H1_SUPPORTED else "NOT SUPPORTED")} at alpha {ALPHA}')

## 4. P1 (24시간 창 AUC 순위)과 P2 (4시간 창 하락 전 포착) — 보조 방향 예측

In [6]:
def in_decline_mask(E_, y_, T_):
    br = E_.breach_utc.astype('int64').to_numpy(); on = E_.onset_1pct_utc.astype('int64').to_numpy()
    pos = np.searchsorted(br, T_ + LEAD_MIN * NS, 'left'); ok = pos < len(br)
    inn = np.zeros(len(T_), bool); inn[ok] = T_[ok] >= on[pos[ok]]
    return inn & (y_ == 1)


if RUN_ANALYSIS:
    t_ = time.time()
    EVM = VALID & (T >= PROS_START.value) & (T < PROS_END.value - (LEAD_MAX + 5) * NS)
    DEC = np.full(len(T), -1, np.int64); okr = EVM & np.isfinite(RV)
    DEC[okr] = np.minimum((pd.Series(RV[okr]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    WEEK = np.where(EVM, (T - PROS_START.value) // (BLOCK_DAYS * 1440 * NS), -1).astype(np.int64); N_WEEKS = int(WEEK[EVM].max()) + 1
    W_BOOT = [np.bincount(np.random.default_rng(SEED + 500 + i).integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS).astype(float) for i in range(N_BOOT)]
    prow, drow = [], []
    for H in sorted({H_P1, H_P2}):
        yH = label_from_catalog(T, E_ALL[H]).astype(int); innH = in_decline_mask(E_ALL[H], yH, T)
        CALM = np.isfinite(F.dd_240.to_numpy(float)) & (F.dd_240.to_numpy(float) < DCL / 4)
        base = EVM & (DEC >= 0) & np.isfinite(S['M0_VF_logit']) & np.isfinite(S['M3_GBM_rich'])
        code = np.where(yH[base] == 1, np.where(innH[base], 2, 1), 0).astype(np.int64)
        pts, bts = {}, {}
        for key in ['M0_VF_logit', 'M3_GBM_rich']:
            P = make_parts(S[key][base], code, CALM[base], WEEK[base], DEC[base]); pts[key] = stats_from(P, np.ones(N_WEEKS))
            bts[key] = {k: np.array([stats_from(P, w)[k] for w in W_BOOT]) for k in ['auc_standard', 'auc_pre', 'auc_pre_within_regime']}
            prow.append(dict(H=H, score=key, n_pre=int((code == 1).sum()), n_in=int((code == 2).sum()), n_neg=int((code == 0).sum()), **{k: pts[key][k] for k in ['auc_standard', 'auc_pre', 'auc_in', 'share_in', 'auc_pre_within_regime']}))
        for k in ['auc_standard', 'auc_pre', 'auc_pre_within_regime']:
            d = pts['M3_GBM_rich'][k] - pts['M0_VF_logit'][k]; bd = bts['M3_GBM_rich'][k] - bts['M0_VF_logit'][k]
            n_def = int(np.isfinite(bd).sum()); defined = np.isfinite(d) and n_def >= MIN_DEFINED_SHARE * N_BOOT
            lo, hi = (np.nanpercentile(bd, [2.5, 97.5]) if defined else (np.nan, np.nan))
            drow.append(dict(H=H, metric=k, gbm_minus_m0=d, lo=lo, hi=hi, draws_defined=n_def, draws=N_BOOT, interval_defined=bool(defined),
                             reading='interval undefined' if not defined else ('GBM ahead' if lo > 0 else ('M0 ahead' if hi < 0 else 'not distinguishable'))))
    PM = pd.DataFrame(prow); DF = pd.DataFrame(drow); PM.to_csv(OUT / '4_per_minute_by_window.csv', index=False); DF.to_csv(OUT / '5_P1_paired_differences.csv', index=False)
    d1 = DF[(DF.H == H_P1) & (DF.metric == 'auc_standard')].iloc[0]; d2 = DF[(DF.H == H_P1) & (DF.metric == 'auc_pre')].iloc[0]
    P1 = ('inconclusive' if not (d1.interval_defined and d2.interval_defined) else
          'supported' if d1.lo > 0 and d2.hi < 0 else 'contradicted' if d1.hi < 0 or d2.lo > 0 else 'inconclusive')
    # P2 at H = 240: events caught pre-onset by M0 vs GBM
    mm0 = match_events(EV[H_P2], ALR['M0_VF_logit']); mm3 = match_events(EV[H_P2], ALR['M3_GBM_rich'])
    p0 = set(mm0.loc[mm0.pre1pct, 'event_id']); p3 = set(mm3.loc[mm3.pre1pct, 'event_id'])
    # paired difference in pre-onset catch rate (M0 - GBM) with a calendar-month block bootstrap over events (notebook 29)
    ev2 = EV[H_P2]; n_ev = len(ev2); c0 = ev2.event_id.isin(p0).to_numpy().astype(float); c3 = ev2.event_id.isin(p3).to_numpy().astype(float)
    rate_diff = float((c0 - c3).mean()) if n_ev else np.nan; months_ev = ev2.breach_utc.dt.strftime('%Y-%m').to_numpy() if n_ev else np.array([])
    rng2 = np.random.default_rng(SEED + 777); mkeys = np.unique(months_ev); bdiff = []
    for _ in range(N_NULL if n_ev else 0):
        pick = rng2.choice(mkeys, len(mkeys)); idx = np.concatenate([np.flatnonzero(months_ev == m_) for m_ in pick])
        bdiff.append((c0[idx] - c3[idx]).mean() if len(idx) else np.nan)
    bdiff = np.asarray(bdiff, float); n_def2 = int(np.isfinite(bdiff).sum()); rd_defined = n_ev > 0 and n_def2 >= MIN_DEFINED_SHARE * N_NULL
    rd_lo, rd_hi = (np.nanpercentile(bdiff, [2.5, 97.5]) if rd_defined else (np.nan, np.nan))
    P2T = dict(H=H_P2, events=n_ev, m0_pre=len(p0), gbm_pre=len(p3), both=len(p0 & p3), m0_only=len(p0 - p3), gbm_only=len(p3 - p0),
               p_m0_better=mcnemar_one_sided(len(p3 - p0), len(p0 - p3)), rate_diff_m0_minus_gbm=rate_diff, rate_diff_lo=rd_lo, rate_diff_hi=rd_hi,
               rate_diff_draws_defined=n_def2, rate_diff_draws=(N_NULL if n_ev else 0), rate_diff_interval_defined=bool(rd_defined), months=len(mkeys))
    P2 = ('not testable' if NO_EVENTS else 'inconclusive' if max(len(p0), len(p3)) < 10 else 'supported' if P2T['p_m0_better'] < ALPHA else 'contradicted' if mcnemar_one_sided(len(p0 - p3), len(p3 - p0)) < ALPHA else 'inconclusive')
    pd.DataFrame([P2T]).to_csv(OUT / '6_P2_paired_events.csv', index=False)
    display(PM.round(3)); display(DF.round(4)); display(pd.DataFrame([P2T]))
    print(f'P1 at H={H_P1}: GBM - M0 standard {d1.gbm_minus_m0:+.3f} [{d1.lo:+.3f}, {d1.hi:+.3f}], pre-onset {d2.gbm_minus_m0:+.3f} [{d2.lo:+.3f}, {d2.hi:+.3f}] -> {P1}')
    print(f'P2 at H={H_P2}: M0 {len(p0)} vs GBM {len(p3)} pre-onset catches (M0 only {len(p0 - p3)}, GBM only {len(p3 - p0)}), p {P2T["p_m0_better"]:.4f}, '
          f'rate difference {rate_diff:+.3f} [{rd_lo:+.3f}, {rd_hi:+.3f}] -> {P2} ({time.time() - t_:.0f}s)')

## 5. 서술: 놓친 사건의 단계 분류, 예산 없는 재생, 리드타임, 요약과 잠금

In [7]:
def replay_calendar(t, elig, B_, cap_, cooldown_min=COOLDOWN_MIN):
    """Python replay of the notebook-5 calendar rule (notebook 27): alerts and, per candidate, the binding constraint (1 annual, 2 monthly, 4 cooldown)."""
    years, months = cal_fields(t)
    idx = np.flatnonzero(elig); reason = np.zeros(len(idx), np.int8); out = []; mcount = ycount = 0; pm = py = -1; last = -(2**62); cd = np.int64(cooldown_min * NS)
    for k, i in enumerate(idx):
        if years[i] != py:
            ycount = 0; py = years[i]
        if months[i] != pm:
            mcount = 0; pm = months[i]
        r = (1 if ycount >= B_ else 0) | (2 if mcount >= cap_ else 0) | (4 if t[i] - last < cd else 0)
        if r:
            reason[k] = r; continue
        out.append(t[i]); last = t[i]; ycount += 1; mcount += 1
    return np.asarray(out, np.int64), idx, reason


if RUN_ANALYSIS:
    POLICIES = {'actual': dict(B=B, cap=MONTH_CAP), 'no_monthly_cap': dict(B=B, cap=B), 'cooldown_only': dict(B=10**9, cap=10**9)}
    srows, crow = [], []
    for key in METHODS16:
        ok_ = SCH & np.isfinite(S[key]); tt = T[ok_]; el = S[key][ok_] >= TH[key]
        rep = {p: replay_calendar(tt, el, kw['B'], kw['cap']) for p, kw in POLICIES.items()}
        if not np.array_equal(rep['actual'][0][rep['actual'][0] >= PROS_START.value], ALR[key]):
            raise RuntimeError(f'GUARD FAIL: Python replay differs from the compiled scheduler ({key})')
        for p in POLICIES:
            al = rep[p][0][rep[p][0] >= PROS_START.value]; c = fast_counts(al, EA)
            crow.append(dict(system=key, policy=p, alerts=len(al), pre=c['pre'], post=c['post']))
        a_act, cidx, creason = rep['actual']; tcand = tt[cidx]
        for e in EV[H_MAIN].itertuples():
            br, on, pk = e.breach_utc.value, e.onset_1pct_utc.value, e.peak_utc.value; w0, w1 = br - LEAD_MAX * NS, br - LEAD_MIN * NS
            sig = (T >= pk - LEAD_MAX * NS) & (T <= br) & np.isfinite(S[key]) & (S[key] >= TH[key]); inw = sig & (T >= w0) & (T <= w1)
            mm = match_events(EV[H_MAIN][EV[H_MAIN].event_id == e.event_id], ALR[key])
            if len(mm):
                stage = 'caught pre-onset' if bool(mm.pre1pct.iloc[0]) else 'caught in-progress'
            elif not sig.any():
                stage = '1 no signal'
            elif not inw.any():
                stage = '2 signal outside the window'
            else:
                rs = creason[(tcand >= w0) & (tcand <= w1)]
                stage = '3 cooldown' if len(rs) and bool(((rs & 4) > 0).all()) else '4 budget'
            srows.append(dict(system=key, event_id=int(e.event_id), breach_utc=e.breach_utc, onset_to_breach_min=(br - on) / NS, stage=stage))
    ST = pd.DataFrame(srows, columns=['system', 'event_id', 'breach_utc', 'onset_to_breach_min', 'stage']); CR = pd.DataFrame(crow)
    ST.to_csv(OUT / '7_event_stages.csv', index=False); CR.to_csv(OUT / '8_policy_replays.csv', index=False)
    if len(ST):
        display(ST.pivot_table(index='system', columns='stage', values='event_id', aggfunc='count', fill_value=0))
    display(CR)
    LT = pd.DataFrame([dict(system=k, pre_onset_catches=len(v), median_lead_min=float(np.median(v)) if v else np.nan,
                            share_within_60_min=float(np.mean(np.asarray(v) <= 60)) if v else np.nan,
                            **{f'catches_at_least_{m}_min_before_onset': int((np.asarray(v) >= m).sum()) for m in LEAD_MARKS},
                            **{f'share_at_least_{m}_min_before_onset': float(np.mean(np.asarray(v) >= m)) if v else np.nan for m in LEAD_MARKS}) for k, v in LEADS.items()])
    LT.to_csv(OUT / '9_lead_times.csv', index=False); display(LT.round(2))
    pd.DataFrame([dict(system=k, lead_min_before_onset=x) for k, v in LEADS.items() for x in v]).to_csv(OUT / '9b_lead_times_per_catch.csv', index=False)
    # data coverage by month and events near the period start
    grid = pd.DatetimeIndex(pd.to_datetime(T, utc=True)); inper = (T >= PROS_START.value) & (T < PROS_END.value)
    cov = pd.DataFrame({'month': grid[inper].strftime('%Y-%m'), 'nan_close': ~np.isfinite(close[inper]), 'invalid': ~VALID[inper]})
    COV = cov.groupby('month').agg(minutes=('invalid', 'size'), nan_close_minutes=('nan_close', 'sum'), invalid_minutes=('invalid', 'sum')).reset_index()
    COV['invalid_share'] = COV.invalid_minutes / COV.minutes
    E_main = E_ALL[H_MAIN]; near = int(((E_main.breach_utc >= PROS_START) & (E_main.breach_utc < PROS_START + pd.Timedelta(minutes=LEAD_MAX))).sum())
    COV.to_csv(OUT / '10_data_coverage.csv', index=False); display(COV)
    print(f'events with a breach in the first {LEAD_MAX} minutes of the period (excluded by the event rule): {near}; invalid minutes in the period: {int((~VALID[inper]).sum()):,} of {int(inper.sum()):,}')
    summary = dict(spec_sha256=SPEC_SHA, evaluated_at=str(pd.Timestamp.now(tz='UTC').floor('s')), period=[str(PROS_START), str(PROS_END)],
                   H1_primary_supported=H1_SUPPORTED, P1=P1, P2=P2, null_tests=NT.to_dict('records'), S3=S3T.to_dict('records'),
                   per_minute=PM.to_dict('records'), P1_differences=DF.to_dict('records'), P2_events=P2T, policy_replays=CR.to_dict('records'),
                   lead_times=LT.to_dict('records'), data_coverage=COV.to_dict('records'), events_excluded_at_period_start=near, frozen_inputs=MANI.to_dict('records'))
    (OUT / 'PROSPECTIVE_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else (bool(o) if isinstance(o, np.bool_) else str(o))))
    LOCK.write_text(f'evaluated {pd.Timestamp.now(tz="UTC")}; spec {SPEC_SHA}\n')
    print('Evaluated once; lock written. H1:', {True: 'SUPPORTED', False: 'NOT SUPPORTED', None: 'NOT TESTABLE'}[H1_SUPPORTED], '| P1:', P1, '| P2:', P2)

## 읽는 법

- **해시 모드:** 첫 셀의 표(`31_FROZEN_INPUTS.csv`)와 `31_PROSPECTIVE_SPEC.json`의 SHA-256, 그리고 이 노트북 파일의 SHA-256을 사전등록문에 그대로 적습니다. 자료는 받지 않습니다.
- **분석 모드:** `2_null_tests.csv`의 M0 행 `p_pre`가 H1입니다. 나머지 행은 보조(S1·S2·S3, Holm 열). `5_P1_paired_differences.csv`와 `6_P2_paired_events.csv`가 P1·P2이고,
  판정은 supported / contradicted / inconclusive 셋 중 하나로만 적습니다. `7_event_stages.csv`·`8_policy_replays.csv`·`9_lead_times.csv`는 서술용입니다.
- 잠금 파일(`PROSPECTIVE_EVALUATED.lock`)이 생기면 이 노트북은 다시 평가하지 않습니다. 24개월 갱신은 `EVAL_MONTHS = 24`로 **새 출력 폴더**를 지정해 서술용으로만 돌립니다.